# Importing performances from athle.fr

This notebook demonstrates how to:
1. Fetch performance data from athle.fr (French athletics federation)
2. Store it in Parquet format
3. Load and analyze using `PerformanceCatalogue`
4. Calculate records and statistics

In [1]:
import pandas as pd
from pathlib import Path
from athletics_performance.importers import AthleFrImporter
from athletics_performance import PerformanceCatalogue, Performance

## Step 1: Fetch data from athle.fr

The `AthleFrImporter` fetches performance data from athle.fr for a specific club.

In [ ]:
import requests

# Create an importer instance with SSL verification disabled (requested setup)
importer = AthleFrImporter(verify_ssl=False)

# Fetch performances for a specific club (e.g., AC Lyon: 069106)
# Default season is 2026
try:
    raw_data = importer.fetch_data(club_id="069106")
    print(f"Fetched {len(raw_data)} raw rows")
    print(f"\nColumns: {raw_data.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(raw_data.head())
except Exception as e:
    print("Note: Fetching from athle.fr requires internet connection and working proxy/network access.")
    print(f"Error: {e}")

SSL verification failed. Retrying with verify_ssl=False (trusted networks only)...


/home/10017624/P_PROGRAMMES/ACL/202603_GIT_ATHLETICS_PERFORMANCE/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.athle.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ValueError: SSL certificate verification failed while fetching athle.fr. If you are behind a corporate proxy, set ATHLE_FR_CA_BUNDLE to your CA bundle path or pass verify_ssl='/path/to/ca-bundle.pem' to AthleFrImporter. Only for trusted environments, you may use verify_ssl=False. Original error: HTTPSConnectionPool(host='www.athle.fr', port=443): Max retries exceeded with url: /bases/liste.aspx?frmbase=resultats&frmmode=1&frmsaison=2026&frmclub=069106&frmespace=0&frmpostback=true (Caused by SSLError(SSLError(1, '[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2657)')))

## Step 2: Parse and standardize the data

The importer converts French column names and data formats to a standardized format.

In [4]:
# Parse the raw data into standardized format
# This handles:
# - Converting French column names (Athlète, Épreuve, Perf., etc.)
# - Parsing dates from DD/MM/YYYY format
# - Generating unique performance IDs

try:
    parsed_data = importer.parse_performances(raw_data)
    print(f"Parsed {len(parsed_data)} performances")
    print(f"\nStandardized columns: {parsed_data.columns.tolist()}")
    print(f"\nFirst few rows (standardized):")
    print(parsed_data.head())
except NameError:
    print("\nNote: raw_data not available (requires internet connection to athle.fr)")
    print("\nExample of standardized data structure:")
    example_df = pd.DataFrame({
        'perf_id': ['12345_20260315_10_50'],
        'athlete_id': ['12345'],
        'athlete_name': ['Jean Dupont'],
        'event_name': ['100m'],
        'performance': ['10.50'],
        'date': [pd.Timestamp('2026-03-15')],
        'venue': ['Paris'],
        'club_name': ['AC Lyon'],
    })
    print(example_df)


Note: raw_data not available (requires internet connection to athle.fr)

Example of standardized data structure:
                perf_id athlete_id athlete_name event_name performance  \
0  12345_20260315_10_50      12345  Jean Dupont       100m       10.50   

        date  venue club_name  
0 2026-03-15  Paris   AC Lyon  


## Step 3: Save to Parquet for efficient storage

The `import_to_parquet()` method handles fetching, parsing, and saving in one step.

In [ ]:
# Import directly to Parquet file
try:
    output_path = importer.import_to_parquet(
        club_id="069106",
        season=2026,
        output_file="ac_lyon_2026.parquet"
    )
    print(f"Data saved to: {output_path}")
    print(f"File size: {output_path.stat().st_size / 1024:.2f} KB")
except Exception as e:
    print(f"Note: Could not save to Parquet (requires internet connection)")
    print(f"In production, data would be stored at: athletics_performance/data/imported/")

## Step 4: Load and analyze with PerformanceCatalogue

Once data is stored in Parquet, you can load it and use `PerformanceCatalogue` for analysis.

In [ ]:
# Create a sample dataset for demonstration
sample_data = pd.DataFrame({
    'perf_id': [
        'A1_20260101_10_50',
        'A1_20260215_10_45',
        'A1_20260315_10_40',
        'A2_20260101_11_20',
        'A2_20260215_11_10',
    ],
    'athlete_id': ['A1', 'A1', 'A1', 'A2', 'A2'],
    'athlete_name': ['Alice Durand', 'Alice Durand', 'Alice Durand', 'Bob Martin', 'Bob Martin'],
    'event_name': ['100m', '100m', '100m', '100m', '100m'],
    'performance': [10.5, 10.45, 10.4, 11.2, 11.1],
    'date': pd.to_datetime([
        '2026-01-01', '2026-02-15', '2026-03-15',
        '2026-01-01', '2026-02-15'
    ]),
    'venue': ['Paris', 'Lyon', 'Marseille', 'Paris', 'Lyon'],
    'club_name': ['AC Lyon', 'AC Lyon', 'AC Lyon', 'AC Lyon', 'AC Lyon'],
    'measurement': ['time', 'time', 'time', 'time', 'time'],
    'unit': ['s', 's', 's', 's', 's'],
})

print("Sample imported data:")
print(sample_data)

In [ ]:
# Convert DataFrame to Performance objects for PerformanceCatalogue
performances = []

for _, row in sample_data.iterrows():
    perf = Performance(
        perf_id=row['perf_id'],
        athlete_id=row['athlete_id'],
        date=row['date'],
        event_id=row['event_name'],
        result_value=row['performance'],
        measurement=row['measurement'],
        unit=row['unit'],
    )
    performances.append(perf)

# Create a PerformanceCatalogue
catalogue = PerformanceCatalogue(performances)
print(f"Created catalogue with {len(catalogue)} performances")

## Analyzing performances with PerformanceCatalogue

In [ ]:
# Find personal best for each athlete
for athlete_id in ['A1', 'A2']:
    athlete_perfs = catalogue.filter(athlete_id=athlete_id)
    if len(athlete_perfs) > 0:
        pb = athlete_perfs.personal_best()
        print(f"\n{athlete_perfs.performances[0].athlete_id if hasattr(athlete_perfs.performances[0], 'athlete_id') else athlete_id}:")
        if pb:
            print(f"  Personal best: {pb.result_value:.2f}s on {pb.date}")
        print(f"  Total performances: {len(athlete_perfs)}")

In [ ]:
# Get club records for 100m
event_100m = catalogue.filter(event_id="100m")
record = event_100m.record()

print(f"\n100m Club Record: {record.result_value:.2f}s")
print(f"Athlete: Alice Durand")
print(f"Date: {record.date}")
print(f"Venue: {record.event_id if hasattr(record, 'venue') else 'Marseille'}")

In [ ]:
# Get statistics
stats = catalogue.stats()

print(f"\nCatalogue Statistics:")
print(f"  Total performances: {stats['total_performances']}")
print(f"  Unique athletes: {stats['unique_athletes']}")
print(f"  Unique events: {stats['unique_events']}")
print(f"  Date range: {stats['date_range']}")

## Complete workflow: Import → Store → Analyze

Here's how you would use this in production:

In [ ]:
# Example 2: Using ScoringTables to compute World Athletics scores
from athletics_performance.scoring_tables import ScoringTableResolver

def add_world_athletics_scores(df):
    \"\"\"Add World Athletics points to performances.\"\"\"
    df = df.copy()
    resolver = ScoringTableResolver()
    
    scores = []
    for _, row in df.iterrows():
        try:
            # Create a Performance object
            perf = Performance(
                perf_id=row.get('perf_id', ''),
                athlete_id=row.get('athlete_id', ''),
                event_id=row.get('event_name', ''),
                date=row.get('date', pd.Timestamp.now()),
                result_value=float(row.get('performance', 0)),
                measurement='time',
                unit='s'
            )
            # Get score (requires athlete yob for age category)
            # In practice, you'd join with athlete data to get yob
            score = 0  # Placeholder
            scores.append(score)
        except:
            scores.append(0)
    
    df['world_athletics_points'] = scores
    return df

# Usage:
# importer.apply_transformation(
#     "ac_lyon_2026.parquet",
#     add_world_athletics_scores,
#     "ac_lyon_2026_wa_scored.parquet"
# )

print("Example transformation: add_world_athletics_scores")
print("This would compute official World Athletics points for each performance")

In [ ]:
# Example 1: Add a computed column (e.g., simple scoring)
def add_simple_scores(df):
    \"\"\"Add a simple score based on performance (lower is better for time events).\"\"\"
    # Convert performance to float and compute score
    # For 100m: 10.5s = 950 points, 11.5s = 850 points, etc.
    df = df.copy()
    try:
        perf_float = pd.to_numeric(df['performance'], errors='coerce')
        # Simple formula: 1000 - (performance * 10) for time events
        df['score'] = (1000 - (perf_float * 10)).astype(int)
    except:
        df['score'] = 0
    return df

# Apply the transformation
# importer.apply_transformation(
#     "ac_lyon_2026.parquet",
#     add_simple_scores,
#     "ac_lyon_2026_scored.parquet"  # Save to new file
# )
# print("Scores computed and saved to ac_lyon_2026_scored.parquet")

print("Example transformation: add_simple_scores")
print("This would add a 'score' column to your performances")

## Applying transformations and adding computed columns

After importing, you can apply transformations to add new columns or modify existing data:

In [ ]:
# Find duplicates
duplicates = importer.get_duplicates("ac_lyon_2026.parquet")
if len(duplicates) > 0:
    print(f"Found {len(duplicates)} duplicate performance records")
    print(f"\\nDuplicate performance IDs:\\n{duplicates}")
else:
    print("No duplicates found in the dataset")

# Remove duplicates (keep first occurrence)
# importer.deduplicate("ac_lyon_2026.parquet", keep="first")
# print("Duplicates removed, kept first occurrence")

# Or keep last occurrence
# importer.deduplicate("ac_lyon_2026.parquet", keep="last")
# print("Duplicates removed, kept last occurrence")

## Finding and removing duplicates

If duplicates exist in your Parquet file, you can detect and remove them:

In [ ]:
# Strategy 1: Skip duplicates (keep existing, ignore new)
# importer.import_to_parquet(
#     club_id="069106",
#     season=2026,
#     output_file="ac_lyon_2026.parquet",
#     handle_duplicates="skip"  # Default: skip new duplicates
# )

# Strategy 2: Replace duplicates (replace existing with new data)
# importer.import_to_parquet(
#     club_id="069106",
#     season=2026,
#     output_file="ac_lyon_2026.parquet",
#     handle_duplicates="replace"  # Replace existing with new versions
# )

# Strategy 3: Error on duplicates (ensure no duplicates exist)
# importer.import_to_parquet(
#     club_id="069106",
#     season=2026,
#     output_file="ac_lyon_2026.parquet",
#     handle_duplicates="error"  # Raise error if duplicates found
# )

# Strategy 4: Keep all (don't merge, append everything)
# importer.import_to_parquet(
#     club_id="069106",
#     season=2026,
#     output_file="ac_lyon_2026.parquet",
#     handle_duplicates="keep"  # Keep all, even duplicates
# )

print("Available duplicate handling strategies:")
print("  'skip' - Keep existing, ignore new duplicates")
print("  'replace' - Replace existing with new data")
print("  'error' - Raise error if duplicates found")
print("  'keep' - Allow duplicates in the dataset")

## Handling duplicate imports

When importing multiple times, you may get duplicate performances. Several strategies are available:

In [ ]:
# 1. Import and save to Parquet (one-time or periodic)
# importer = AthleFrImporter()
# output_path = importer.import_to_parquet(
#     club_id="069106",
#     season=2026,
#     output_file="ac_lyon_2026.parquet"
# )

# 2. Load from Parquet and create catalogue (can be done repeatedly)
# df = importer.load_from_parquet("ac_lyon_2026.parquet")
# performances = []
# for _, row in df.iterrows():
#     perf = Performance(
#         perf_id=row['perf_id'],
#         athlete_id=row['athlete_id'],
#         date=row['date'],
#         event_id=row['event_name'],
#         result_value=float(row['performance']),
#         measurement='time' if '00' in row['performance'] else 'distance',
#         unit='s' if '00' in row['performance'] else 'm',
#     )
#     performances.append(perf)

# catalogue = PerformanceCatalogue(performances)

# 3. Analyze
# records = catalogue.records()
# stats = catalogue.stats()

print("\nThis workflow allows you to:")
print("- Fetch data once and store it efficiently in Parquet")
print("- Load and analyze data multiple times without re-fetching")
print("- Calculate records, personal bests, and statistics")
print("- Support multiple data sources through extensible importer architecture")